# 授業方略実験ノートブック

最終更新: 2026-07-13 15:25 JST

このノートブックでは、教師AIが授業手法を考える前段階を確認します。生徒AIの発話を作り、伝達AIが観察し、教師AI用のコンテキストを作成して、次の授業方略を選びます。

## 1. GitHubから最新版を取得

Colabでは最初にこのセルを実行します。まだcloneしていない場合はcloneし、すでにclone済みの場合は最新版をpullします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline
!grep -n "summarize_classroom" src/observer/trait_classifier.py
!grep -n "classroom_observation" src/teacher/context_builder.py
!grep -n "TeacherBeliefManager" src/teacher/belief_manager.py

## 2. 依存関係をインストール

In [ ]:
!pip install -q -r requirements.txt

## 3. 実験設定

軽く確認する場合は `USE_MOCK_MODEL = True` のまま使います。実LLMで生徒発話を生成したい場合は `False` に変更します。

In [ ]:
STUDENT_ID = "S002"
USE_MOCK_MODEL = True
TEACHER_MESSAGE = "x + 3 = 8 を解いてみましょう。+3を反対側へ移すと、符号はどうなりますか。"

print("student_id:", STUDENT_ID)
print("use_mock_model:", USE_MOCK_MODEL)
print("teacher_message:", TEACHER_MESSAGE)

## 4. 生徒AIの発話を生成

ここでは授業中の対話として生徒発話を生成します。実験を繰り返しやすくするため、標準では知識状態を更新しません。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=USE_MOCK_MODEL)
student_record = sim.respond(
    STUDENT_ID,
    TEACHER_MESSAGE,
    update_knowledge=False,
)
student_utterance = student_record["answer"]

print(student_utterance)

## 5. 伝達AIが発話を観察

伝達AIが、生徒発話から見える個人特徴を推定し、教師AIに渡す注意点を作ります。

In [ ]:
import importlib
from pprint import pprint

import src.observer.trait_classifier as trait_classifier

importlib.reload(trait_classifier)
CommunicationAI = trait_classifier.CommunicationAI

communication_ai = CommunicationAI()
print("has summarize_classroom:", hasattr(communication_ai, "summarize_classroom"))
observation = communication_ai.classify_utterance(
    utterance=student_utterance,
    student_id=STUDENT_ID,
).to_dict()

pprint(observation)

## 6. 複数生徒を実際に反応させて要約

S001、S002、S003を同じ教師発話に反応させます。伝達AIには内部状態ではなく、授業中に観察できる発話、正誤、反応時間などだけを渡します。

In [ ]:
import importlib
import time
from pprint import pprint

from src.grader import LinearEquationGrader
import src.observer.observation_filter as observation_filter

importlib.reload(observation_filter)
build_observable_event = observation_filter.build_observable_event
events_to_communication_rows = observation_filter.events_to_communication_rows

CLASS_STUDENT_IDS = ["S001", "S002", "S003"]
EXPECTED_ANSWER = "x = 5"
grader = LinearEquationGrader()

classroom_records = []
observable_events = []

for sid in CLASS_STUDENT_IDS:
    started = time.perf_counter()
    record = sim.respond(sid, TEACHER_MESSAGE, update_knowledge=False)
    response_time_sec = round(time.perf_counter() - started, 2)
    utterance = record["answer"]
    grade = grader.grade(EXPECTED_ANSWER, utterance)
    event = build_observable_event(
        lesson_id="L001",
        teacher_id="T001",
        student_id=sid,
        teacher_prompt=TEACHER_MESSAGE,
        utterance=utterance,
        answer=utterance,
        is_correct=grade["is_correct"],
        response_time_sec=response_time_sec,
    ).to_dict()
    classroom_records.append({"student_id": sid, "utterance": utterance, "grade": grade})
    observable_events.append(event)

classroom_rows = events_to_communication_rows(observable_events)
classroom_observation = communication_ai.summarize_classroom(classroom_rows).to_dict()

pprint({
    "classroom_records": classroom_records,
    "student_count": classroom_observation["student_count"],
    "profile_counts": classroom_observation["profile_counts"],
    "trait_level_counts": classroom_observation["trait_level_counts"],
    "priority_students": classroom_observation["priority_students"],
    "classroom_summary": classroom_observation["classroom_summary"],
    "recommended_teacher_actions": classroom_observation["recommended_teacher_actions"],
})

## 7. 複数生徒の教師側推定を更新して表で確認

各生徒の `teacher_belief` を、観察イベントと伝達AIの個別分類結果から更新します。表では推定理解度、confidence、性格・心理推定の変化を見ます。

In [ ]:
import importlib
import pandas as pd
from pprint import pprint

import src.teacher.belief_manager as belief_manager

importlib.reload(belief_manager)

TeacherBeliefManager = belief_manager.TeacherBeliefManager

belief_manager_instance = TeacherBeliefManager()
teacher_beliefs = belief_manager_instance.update_many(
    teacher_id="T001",
    observations=observable_events,
    communication_results=classroom_observation["individual_results"],
)
teacher_belief = teacher_beliefs[STUDENT_ID]

belief_rows = []
for sid, belief in teacher_beliefs.items():
    knowledge = belief["estimated_knowledge"]["linear_equation"]
    traits = belief["estimated_traits"]
    event = next(item for item in observable_events if item["student_id"] == sid)
    belief_rows.append({
        "student_id": sid,
        "is_correct": event["is_correct"],
        "response_time_sec": event["response_time_sec"],
        "estimated_score": knowledge["score"],
        "score_confidence": knowledge["confidence"],
        "self_efficacy": traits["self_efficacy"]["level"],
        "self_efficacy_conf": traits["self_efficacy"]["confidence"],
        "question_tendency": traits["question_tendency"]["level"],
        "motivation": traits["motivation"]["level"],
        "neuroticism": traits["neuroticism"]["level"],
        "evidence_count": len(belief["evidence_history"]),
    })

belief_table = pd.DataFrame(belief_rows).sort_values("student_id")
display(belief_table)
pprint(teacher_beliefs)

## 8. 教師AI用コンテキストを作成

教師AI用コンテキストには、生徒状態、単元目標、直近の生徒発話、伝達AIの個別観察、クラス全体の要約、教師側の推定値、次に出せる問題が入ります。

In [ ]:
import importlib
from pprint import pprint

from src.state_manager import StateManager
import src.teacher.context_builder as teacher_context_builder

importlib.reload(teacher_context_builder)
TeacherContextBuilder = teacher_context_builder.TeacherContextBuilder

state_manager = StateManager()
student_state = state_manager.load_student(STUDENT_ID)

context_builder = TeacherContextBuilder()
teacher_context = context_builder.build_context(
    student_state=student_state,
    recent_student_utterance=student_utterance,
    communication_observation=observation,
    classroom_observation=classroom_observation,
    teacher_belief=teacher_belief,
)

pprint({
    "target_skill": teacher_context["target_skill"],
    "lesson_goal": teacher_context["lesson_goal"],
    "student_state_summary": teacher_context["student_state_summary"],
    "communication_ai_observation": teacher_context["communication_ai_observation"],
})

## 9. 授業方略を選択

現在のMVPでは、判断理由を確認しやすいようにルールベースで授業方略を選びます。将来的には同じコンテキストをLLM教師に渡せます。

In [ ]:
from pprint import pprint

from src.teacher import RuleBasedTeachingStrategySelector

selector = RuleBasedTeachingStrategySelector()
teacher_decision = selector.select_strategy(teacher_context)

pprint(teacher_decision)

## 10. 最新結果を保存

あとで比較できるように、教師AI用コンテキストと授業方略の判断結果をJSONで保存します。

In [ ]:
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)

context_path = output_dir / "teacher_context_latest.json"
decision_path = output_dir / "teaching_strategy_decision_latest.json"
events_path = output_dir / "observable_events_latest.json"
beliefs_path = output_dir / "teacher_beliefs_latest.json"
belief_table_path = output_dir / "teacher_belief_table_latest.csv"

context_path.write_text(json.dumps(teacher_context, ensure_ascii=False, indent=2), encoding="utf-8")
decision_path.write_text(json.dumps(teacher_decision, ensure_ascii=False, indent=2), encoding="utf-8")
events_path.write_text(json.dumps(observable_events, ensure_ascii=False, indent=2), encoding="utf-8")
beliefs_path.write_text(json.dumps(teacher_beliefs, ensure_ascii=False, indent=2), encoding="utf-8")
belief_table.to_csv(belief_table_path, index=False)

print("saved:", context_path)
print("saved:", decision_path)
print("saved:", events_path)
print("saved:", beliefs_path)
print("saved:", belief_table_path)